# Multiple Linear Regression Using `SGDRegressor`

This notebook uses Scikit-Learn's `SGDRegressor` for **multiple input features** and one output `y`.

The model is still:

`ŷ = w1*x1 + w2*x2 + ... + wn*xn + b`

The difference from `LinearRegression` is that the parameters are learned using stochastic gradient descent, rather than the exact ordinary-least-squares solution.

This is the library-model counterpart of the multi-feature gradient-descent mathematics implemented from scratch in the other notebook.

## 1. Imports

`SGDRegressor` provides a linear regression model trained using stochastic gradient descent, and generalizes to any number of input features automatically.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler


## 2. Create the dataset

We have three input features — `Hours_Studied`, `Sleep_Hours`, `Attendance_Percentage` — and one output, `Marks`.

In [ ]:
df = pd.DataFrame({
    "Hours_Studied":          [2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
    "Sleep_Hours":            [6, 7, 6, 8, 7, 6, 8, 7, 8, 6],
    "Attendance_Percentage":  [60, 65, 70, 72, 78, 80, 85, 88, 90, 95],
    "Marks":                  [42, 48, 51, 58, 63, 66, 74, 78, 85, 88]
})

df

## 3. Separate input and target

`X` is kept as a 2D DataFrame with all three feature columns.

`y` is the one-dimensional target.

In [ ]:
X = df[["Hours_Studied", "Sleep_Hours", "Attendance_Percentage"]]
y = df["Marks"]

print("X:")
print(X)

print("\ny:")
print(y)


## 4. Feature scaling

`SGDRegressor` is sensitive to the scale of input features — features with larger raw values (like `Attendance_Percentage`) can dominate the gradient updates over features with smaller raw values (like `Hours_Studied`) if left unscaled.

We use Scikit-Learn's `StandardScaler` to standardize every feature to mean 0 and standard deviation 1, the same normalization used in the from-scratch gradient descent notebook.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled X (first 3 rows):")
print(X_scaled[:3])


## 5. Create the SGDRegressor model

Important parameters:

- `max_iter`: maximum number of passes/iterations over the training data.
- `learning_rate="constant"`: keeps the learning rate fixed.
- `eta0=0.01`: initial/fixed learning-rate value for this schedule.
- `random_state=42`: makes the random behavior reproducible.

The model has been created, but the weights and bias have not been learned yet.

In [ ]:
model = SGDRegressor(
    max_iter=1000,
    learning_rate="constant",
    eta0=0.01,
    random_state=42
)


## 6. Train using stochastic gradient descent

`fit(X_scaled, y)` starts the optimization process.

The model repeatedly updates its weight vector and bias using gradient-based updates until the configured training process stops.

In [ ]:
model.fit(X_scaled, y)


## 7. Inspect training information

`n_iter_` tells us how many passes over the training data were actually used.

`n_features_in_` tells us how many input features were used — now 3 instead of 1.

In [ ]:
print("Iterations used:", model.n_iter_)
print("Number of features:", model.n_features_in_)


## 8. Extract the learned parameters

`coef_` now contains **one weight per feature** (an array of length 3), learned on the scaled features.

`intercept_[0]` is the learned bias/intercept `b`.

In [ ]:
w = model.coef_
b = model.intercept_[0]

for feature, weight in zip(X.columns, w):
    print(f"Weight for {feature} (scaled): {weight:.4f}")

print("Bias (b):", b)


## 9. Make predictions

The trained model uses the learned weights and bias, applied to the **scaled** features, to calculate predictions:

`ŷ = Xw + b`

In [ ]:
predictions = model.predict(X_scaled)

df["Predicted"] = predictions

df

## 10. Visualize predicted vs actual values

With multiple features there is no single regression line to draw, so we compare predicted and actual Marks against the diagonal `ŷ = y` line.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y, predictions, label="Predictions")

min_val = min(y.min(), predictions.min()) - 2
max_val = max(y.max(), predictions.max()) + 2
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", label="Perfect Prediction (y = ŷ)")

plt.xlabel("Actual Marks")
plt.ylabel("Predicted Marks")
plt.title("Predicted vs Actual — SGDRegressor")
plt.legend()
plt.grid(True)
plt.show()


## 11. Calculate residuals

A residual is:

`residual = y - ŷ`

This shows the prediction error for every training observation.

In [ ]:
df["Residual"] = y - predictions

df

## 12. Visualize residuals

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(range(len(df)), df["Residual"])
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Training example index")
plt.ylabel("Residual (y - ŷ)")
plt.title("Residuals After SGD Training")
plt.grid(True)
plt.show()


## 13. Mean Squared Error (MSE)

MSE measures the average squared prediction error.

`MSE = mean((y - ŷ)²)`

In [ ]:
mse = np.mean((y - predictions) ** 2)

print("MSE:", mse)


## 14. Root Mean Squared Error (RMSE)

RMSE is the square root of MSE and has the same units as the target.

In [ ]:
rmse = np.sqrt(mse)

print("RMSE:", rmse)


## 15. Mean Absolute Error (MAE)

MAE is the average absolute prediction error.

In [ ]:
mae = np.mean(np.abs(y - predictions))

print("MAE:", mae)


## 16. R² score

`score(X, y)` returns the R² score for the fitted regressor.

It measures how well the fitted model explains the variation in the target, using all input features together.

In [ ]:
r2 = model.score(X_scaled, y)

print("R²:", r2)


## 17. Predict a new input

New inputs must be scaled with the **same** `scaler` fitted on the training data before being passed to the model.

In [ ]:
new_X = pd.DataFrame({
    "Hours_Studied": [9],
    "Sleep_Hours": [7],
    "Attendance_Percentage": [82]
})

new_X_scaled = scaler.transform(new_X)
new_prediction = model.predict(new_X_scaled)

print("Input:")
print(new_X)
print("\nPredicted Marks:", new_prediction[0])


## 18. Understand the relationship with gradient descent

The mathematical notebook implemented the vectorized gradient updates explicitly:

`w := w - α(∂J/∂w)`

`b := b - α(∂J/∂b)`

Here, `SGDRegressor` performs the same kind of optimization internally, across all features and the bias at once.

You provide the (scaled) data and configuration such as the learning-rate schedule and iteration limit, and Scikit-Learn performs the parameter updates.

## 19. Important SGD parameters

The main parameters worth understanding for multi-feature regression are the same as in the single-variable case:

`max_iter` → maximum number of passes over the training data.

`learning_rate` → controls the learning-rate schedule.

`eta0` → initial learning-rate value.

`tol` → stopping criterion used by the estimator when convergence is checked.

`random_state` → makes the randomized behavior reproducible.

With multiple features, feature scaling (Section 4) becomes far more important for reliable convergence than it is with a single feature.

In [ ]:
print("Final iterations:", model.n_iter_)
print("Final weights (w):", w)
print("Final bias (b):", b)


## 20. Final evaluation summary

In [ ]:
print("w (weights, scaled features):", w)
print(f"b (bias)   : {b:.6f}")
print(f"MSE : {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE : {mae:.6f}")
print(f"R²  : {r2:.6f}")
